In [ ]:
link server AI
https://colab.research.google.com/drive/1_oKGufYDNKy2-gy_uh7sG0xIYKVj_dH1?usp=sharing

In [ ]:
#Cài đặt môi trường
!pip install aiortc python-socketio[client] nest_asyncio

In [ ]:

# Logic AI WebRTC
import asyncio
import socketio
import nest_asyncio
from aiortc import RTCPeerConnection, RTCSessionDescription
from aiortc.sdp import candidate_from_sdp

nest_asyncio.apply()
sio = socketio.AsyncClient()
pc = RTCPeerConnection()

@sio.on("signal")
async def on_signal(data):
    if "sdp" in data:
        offer = RTCSessionDescription(data["sdp"]["sdp"], data["sdp"]["type"])
        await pc.setRemoteDescription(offer)
        if offer.type == "offer":
            answer = await pc.createAnswer()
            await pc.setLocalDescription(answer)
            await sio.emit("signal", {"sdp": {"sdp": pc.localDescription.sdp, "type": pc.localDescription.type}})
            print("✅ Đã bắt tay (Handshake) với Browser")

    elif "candidate" in data:
        cand = data["candidate"]
        if cand and cand.get("candidate"):
            candidate = candidate_from_sdp(cand["candidate"])
            candidate.sdpMid = cand.get("sdpMid")
            candidate.sdpMLineIndex = cand.get("sdpMLineIndex")
            await pc.addIceCandidate(candidate)

@pc.on("track")
def on_track(track):
    print(f"🎤 Nhận được luồng {track.kind} - Đang sẵn sàng để AI dịch thuật...")

async def start_ai():
    await sio.connect("http://xomnhala.ddns.net:3000")
    await sio.wait()

asyncio.run(start_ai())